<a href="https://colab.research.google.com/github/volodymyr-holovan/melbourne-housing-snapshot/blob/main/Analysis/melbourne_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1. Connecting Data and Importing Libraries

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pls
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Setting plot style
sns.set_theme (style="whitegrid", palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

#loading data
DATA_URL = 'https://raw.githubusercontent.com/volodymyr-holovan/melbourne-housing-snapshot/refs/heads/main/Data/melb_data.csv'
df = pd.read_csv(DATA_URL)

print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

Loaded: 13580 rows, 21 columns


## Step 2. Initial data review

In [11]:
df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [14]:
df.describe().round(1)

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.0,13580.0,13580.0,13580.0,13580.0,13580.0,13518.0,13580.0,7130.0,8205.0,13580.0,13580.0,13580.0
mean,2.9,1075684.1,10.1,3105.3,2.9,1.5,1.6,558.4,152.0,1964.7,-37.8,145.0,7454.4
std,1.0,639310.7,5.9,90.7,1.0,0.7,1.0,3990.7,541.0,37.3,0.1,0.1,4378.6
min,1.0,85000.0,0.0,3000.0,0.0,0.0,0.0,0.0,0.0,1196.0,-38.2,144.4,249.0
25%,2.0,650000.0,6.1,3044.0,2.0,1.0,1.0,177.0,93.0,1940.0,-37.9,144.9,4380.0
50%,3.0,903000.0,9.2,3084.0,3.0,1.0,2.0,440.0,126.0,1970.0,-37.8,145.0,6555.0
75%,3.0,1330000.0,13.0,3148.0,3.0,2.0,2.0,651.0,174.0,1999.0,-37.8,145.1,10331.0
max,10.0,9000000.0,48.1,3977.0,20.0,8.0,10.0,433014.0,44515.0,2018.0,-37.4,145.5,21650.0


In [16]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_report = pd.DataFrame({'Missing': missing, '% from all': missing_pct})
missing_report = missing_report[missing_report['Missing'] > 0].sort_values('Missing', ascending=False)
missing_report

,Missing,% from all
BuildingArea,6450,47.5
YearBuilt,5375,39.6
CouncilArea,1369,10.1
Car,62,0.5


### 🔍 Findings from the Initial Review

| Issue | What We See |
|---|---|
| **Date** | `Date` was parsed as an `object` (string) rather than a date. Format `DD/MM/YYYY`. |
| **Missing Values** | `BuildingArea` (47%), `YearBuilt` (40%), `CouncilArea` (10%) — many. `Car` — few. |
| **YearBuilt** | Minimum `1196` — an obvious error. A year before 1800 is physically impossible. |
| **Landsize = 0** | 1,939 records with zero lot area. These are not data, but missing values—there is no such thing as zero land. |
| **Bedroom2 vs Rooms** | Correlation ~0.94—almost a duplicate. Let’s keep only `Rooms`. |
| **Duplicate rows** | 0 — clean. |
| **Suburb** | 314 unique values — encoding via OHE is not practical (too broad). Frequency encoding is better. |